# LeetCode 1217 — Minimum Cost to Move Chips to The Same Position bằng Python

**Link đề:** https://leetcode.com/problems/minimum-cost-to-move-chips-to-the-same-position/
**Độ khó:** Easy · **Chủ đề:** Array, Math, Greedy

```python
class Solution:
    def minCostToMoveChips(self, position: list[int]) -> int:
        ...
```

## Mục tiêu học
1. Tập **đọc kỹ luật chơi** và phát hiện **bất biến chẵn/lẻ (parity)** — mấu chốt biến bài "di chuyển" thành bài "đếm".
2. Nắm toán tử `%`, kiểm tra chẵn lẻ, `abs`, `sum` với generator, `min`.
3. Đi từ brute force (thử mọi vị trí đích) → nhận ra chỉ cần thử **2 vị trí** → công thức `min(số chẵn, số lẻ)`.
4. Trace, chứng minh, test đối chiếu với một bộ giải BFS/Dijkstra thật sự trên trục số nhỏ.
5. Bài tập TODO + gợi ý 3 mức + đáp án ở cuối.

## 1. Đề bài và ví dụ

Có `n` con chip, chip thứ `i` nằm ở vị trí `position[i]` trên trục số. Ta muốn dồn **mọi chip về cùng một vị trí**. Mỗi bước, có thể đổi vị trí chip thứ `i` từ `position[i]` sang:

- `position[i] + 2` hoặc `position[i] - 2` với **chi phí 0**;
- `position[i] + 1` hoặc `position[i] - 1` với **chi phí 1**.

Trả về chi phí **nhỏ nhất** để đưa tất cả chip về cùng một vị trí.

| `position` | Kết quả | Giải thích |
|---|:---:|---|
| `[1, 2, 3]` | `1` | chip ở 3 → 1 (bước 2, miễn phí); chip ở 2 → 1 (bước 1, tốn 1) |
| `[2, 2, 2, 3, 3]` | `2` | hai chip ở 3 → 2, mỗi chip tốn 1 |
| `[1, 1000000000]` | `1` | khoảng cách lớn nhưng chỉ tốn 1! |

**Constraints:** `1 <= position.length <= 100`; `1 <= position[i] <= 10^9`.

Lưu ý: "position" là **vị trí** của từng chip, nhiều chip có thể ở cùng vị trí (ví dụ 2).

## 2. Nền tảng Python cần biết

### 2.1. Toán tử `%` và chẵn/lẻ

`a % 2` là số dư khi chia `a` cho 2: bằng `0` nếu `a` chẵn, `1` nếu `a` lẻ. Đây gọi là **parity** (tính chẵn lẻ) của `a`.

In [1]:
for a in [1, 2, 3, 10, 1000000000, 999999999]:
    print(f"{a:>10} % 2 = {a % 2} -> {'chẵn' if a % 2 == 0 else 'lẻ'}")

         1 % 2 = 1 -> lẻ
         2 % 2 = 0 -> chẵn
         3 % 2 = 1 -> lẻ
        10 % 2 = 0 -> chẵn
1000000000 % 2 = 0 -> chẵn
 999999999 % 2 = 1 -> lẻ


### 2.2. Cộng/trừ 2 không đổi chẵn lẻ, cộng/trừ 1 thì đổi

Nếu `a` chẵn thì `a ± 2` vẫn chẵn; `a ± 1` thành lẻ. Kiểm chứng:

In [2]:
for a in [4, 7]:
    print(f"a={a} (parity {a % 2}) | a+2={a + 2} (parity {(a + 2) % 2}) | a-2={a - 2} (parity {(a - 2) % 2}) | a+1={a + 1} (parity {(a + 1) % 2})")

a=4 (parity 0) | a+2=6 (parity 0) | a-2=2 (parity 0) | a+1=5 (parity 1)
a=7 (parity 1) | a+2=9 (parity 1) | a-2=5 (parity 1) | a+1=8 (parity 0)


### 2.3. `sum` với generator và đếm theo điều kiện

`sum(1 for p in position if p % 2 == 0)` đếm số phần tử chẵn. Cũng có thể viết `sum(p % 2 == 0 for p in position)` vì `True` được tính là `1`, `False` là `0`.

In [3]:
position = [2, 2, 2, 3, 3]
print("số chẵn:", sum(1 for p in position if p % 2 == 0))
print("số lẻ  :", sum(p % 2 for p in position))          # p % 2 = 1 khi lẻ
print("True + True + False =", True + True + False)

số chẵn: 3
số lẻ  : 2
True + True + False = 2


### 2.4. `abs` và khoảng cách trên trục số

Khoảng cách giữa `p` và `t` là `abs(p - t)`. Ta sẽ cần nó trong brute force.

In [4]:
print(abs(3 - 1), abs(1 - 3), abs(1000000000 - 1))

2 2 999999999


## 3. Cách 1 — Brute force: thử từng vị trí đích

Ý tưởng tự nhiên: chọn một vị trí đích `t`, tính chi phí đưa từng chip về `t`, lấy `t` tốt nhất.

Chi phí đưa một chip từ `p` về `t`: đi khoảng cách `d = abs(p - t)`.
- Dùng tối đa các bước 2 (miễn phí): `d // 2` bước.
- Nếu `d` lẻ, còn dư đúng 1 → tốn **1** bước 1.

Vậy chi phí một chip = `d % 2`. Ta thử `t` là mỗi vị trí đang có chip (đích tối ưu luôn có thể chọn trùng một vị trí chip — sẽ chứng minh ở mục 8).

In [5]:
def cost_one_chip(p: int, t: int) -> int:
    d = abs(p - t)
    return d % 2                     # d//2 bước miễn phí + (d % 2) bước tốn 1

def min_cost_brute(position: list[int]) -> int:
    best = float("inf")
    for t in position:
        total = sum(cost_one_chip(p, t) for p in position)
        best = min(best, total)
    return best

print(cost_one_chip(3, 1), cost_one_chip(2, 1), cost_one_chip(1000000000, 1))
for pos in [[1, 2, 3], [2, 2, 2, 3, 3], [1, 1000000000]]:
    print(pos, "->", min_cost_brute(pos))

0 1 1
[1, 2, 3] -> 1
[2, 2, 2, 3, 3] -> 2
[1, 1000000000] -> 1


**Chi phí:** `n` vị trí đích × `n` chip = `O(n²)`. Với `n ≤ 100` thì vẫn ổn, nhưng nhìn kỹ hơn sẽ thấy ta đang lặp lại công việc vô ích.

## 4. Nút thắt và insight — chỉ có **chẵn lẻ** là quan trọng

Quan sát bảng chi phí khi thử các đích khác nhau cho `[2, 2, 2, 3, 3]`:

In [6]:
position = [2, 2, 2, 3, 3]
for t in range(1, 7):
    costs = [cost_one_chip(p, t) for p in position]
    print(f"đích t={t} (parity {t % 2}): chi phí từng chip={costs} -> tổng {sum(costs)}")

đích t=1 (parity 1): chi phí từng chip=[1, 1, 1, 0, 0] -> tổng 3
đích t=2 (parity 0): chi phí từng chip=[0, 0, 0, 1, 1] -> tổng 2
đích t=3 (parity 1): chi phí từng chip=[1, 1, 1, 0, 0] -> tổng 3
đích t=4 (parity 0): chi phí từng chip=[0, 0, 0, 1, 1] -> tổng 2
đích t=5 (parity 1): chi phí từng chip=[1, 1, 1, 0, 0] -> tổng 3
đích t=6 (parity 0): chi phí từng chip=[0, 0, 0, 1, 1] -> tổng 2


Thấy ngay: **mọi đích chẵn cho cùng tổng, mọi đích lẻ cho cùng tổng**. Lý do:

- `abs(p - t) % 2 == 0` ⇔ `p` và `t` **cùng** chẵn lẻ.
- Nên chip **cùng parity** với đích: miễn phí. Chip **khác parity**: tốn đúng 1.

Do đó:
- Dồn về một vị trí **chẵn**: chi phí = số chip ở vị trí **lẻ**.
- Dồn về một vị trí **lẻ**: chi phí = số chip ở vị trí **chẵn**.

Đáp án = `min(số chip chẵn, số chip lẻ)`. Khoảng cách thật (dù là `10^9`) **không quan trọng**!

## 5. Lời giải chính — giảng từng đoạn rồi ghép

**Đoạn 1 — đếm chip ở vị trí chẵn và lẻ:**

In [7]:
position = [1, 2, 3]
even = odd = 0
for p in position:
    if p % 2 == 0:
        even += 1
    else:
        odd += 1
print("even =", even, "| odd =", odd)

even = 1 | odd = 2


**Đoạn 2 — chọn nhóm nhỏ hơn để "trả phí":** nhóm lớn hơn đứng yên về mặt chi phí (đích cùng parity với nó), nhóm nhỏ hơn mỗi chip tốn 1.

In [8]:
print("chi phí =", min(even, odd))

chi phí = 1


**Ghép lại:**

In [9]:
def min_cost_parity(position: list[int]) -> int:
    even = odd = 0
    for p in position:
        if p % 2 == 0:
            even += 1
        else:
            odd += 1
    return min(even, odd)

for pos in [[1, 2, 3], [2, 2, 2, 3, 3], [1, 1000000000]]:
    print(pos, "->", min_cost_parity(pos))

[1, 2, 3] -> 1
[2, 2, 2, 3, 3] -> 2
[1, 1000000000] -> 1


### Cách 3 — Chỉ thử 2 đích: `t = 1` và `t = 2`

Phiên bản "cầu nối" giữa brute force và công thức: vì mọi đích cùng parity cho cùng chi phí, chỉ cần đại diện một đích lẻ (1) và một đích chẵn (2).

In [10]:
def min_cost_two_targets(position: list[int]) -> int:
    return min(sum(abs(p - t) % 2 for p in position) for t in (1, 2))

for pos in [[1, 2, 3], [2, 2, 2, 3, 3], [1, 1000000000]]:
    print(pos, "->", min_cost_two_targets(pos))

[1, 2, 3] -> 1
[2, 2, 2, 3, 3] -> 2
[1, 1000000000] -> 1


### Cách 4 — Một dòng với `sum`

Số chip lẻ = `sum(p % 2 for p in position)` (cộng các số dư 0/1); số chip chẵn = `n - odd`.

In [11]:
def min_cost_oneliner(position: list[int]) -> int:
    odd = sum(p % 2 for p in position)
    return min(odd, len(position) - odd)

for pos in [[1, 2, 3], [2, 2, 2, 3, 3], [1, 1000000000]]:
    print(pos, "->", min_cost_oneliner(pos))

[1, 2, 3] -> 1
[2, 2, 2, 3, 3] -> 2
[1, 1000000000] -> 1


## 6. TRACE từng bước

Ví dụ `position = [2, 2, 2, 3, 3]`:

| i | p | p % 2 | Nhóm | even | odd |
|:---:|---:|:---:|---|:---:|:---:|
| 0 | 2 | 0 | chẵn | 1 | 0 |
| 1 | 2 | 0 | chẵn | 2 | 0 |
| 2 | 2 | 0 | chẵn | 3 | 0 |
| 3 | 3 | 1 | lẻ | 3 | 1 |
| 4 | 3 | 1 | lẻ | 3 | 2 |

Kết luận: dồn về vị trí chẵn (ví dụ 2) → 2 chip lẻ trả 1 mỗi chip → **2**.

In [12]:
def trace_chips(position: list[int]) -> int:
    even = odd = 0
    for i, p in enumerate(position):
        if p % 2 == 0:
            even += 1
            group = "chẵn"
        else:
            odd += 1
            group = "lẻ"
        print(f"i={i} p={p:>10} p%2={p % 2} -> nhóm {group:4} | even={even} odd={odd}")
    if even >= odd:
        print(f"Dồn về một vị trí CHẴN: {odd} chip lẻ mỗi chip tốn 1 -> {odd}")
    else:
        print(f"Dồn về một vị trí LẺ: {even} chip chẵn mỗi chip tốn 1 -> {even}")
    return min(even, odd)

assert trace_chips([2, 2, 2, 3, 3]) == 2

i=0 p=         2 p%2=0 -> nhóm chẵn | even=1 odd=0
i=1 p=         2 p%2=0 -> nhóm chẵn | even=2 odd=0
i=2 p=         2 p%2=0 -> nhóm chẵn | even=3 odd=0
i=3 p=         3 p%2=1 -> nhóm lẻ   | even=3 odd=1
i=4 p=         3 p%2=1 -> nhóm lẻ   | even=3 odd=2
Dồn về một vị trí CHẴN: 2 chip lẻ mỗi chip tốn 1 -> 2


Trace thêm ví dụ có khoảng cách khổng lồ để thấy khoảng cách không ảnh hưởng:

In [13]:
assert trace_chips([1, 1000000000]) == 1
print()
assert trace_chips([1, 3, 5, 999999999]) == 0

i=0 p=         1 p%2=1 -> nhóm lẻ   | even=0 odd=1
i=1 p=1000000000 p%2=0 -> nhóm chẵn | even=1 odd=1
Dồn về một vị trí CHẴN: 1 chip lẻ mỗi chip tốn 1 -> 1

i=0 p=         1 p%2=1 -> nhóm lẻ   | even=0 odd=1
i=1 p=         3 p%2=1 -> nhóm lẻ   | even=0 odd=2
i=2 p=         5 p%2=1 -> nhóm lẻ   | even=0 odd=3
i=3 p= 999999999 p%2=1 -> nhóm lẻ   | even=0 odd=4
Dồn về một vị trí LẺ: 0 chip chẵn mỗi chip tốn 1 -> 0


Mô phỏng **thật** đường đi của một chip để tin rằng "đi 2 miễn phí" đúng là đi xa tuỳ ý: chip ở `9` về `2` (khác parity):

In [14]:
def simulate_path(p: int, t: int) -> int:
    cost = 0
    path = [p]
    while abs(p - t) >= 2:                  # còn xa ít nhất 2 -> bước 2 miễn phí
        p += 2 if t > p else -2
        path.append(p)
    if p != t:                              # còn cách đúng 1 -> bước 1 tốn phí
        p = t
        cost += 1
        path.append(p)
    print("đường đi:", " -> ".join(map(str, path)), "| chi phí:", cost)
    return cost

assert simulate_path(9, 2) == 1
assert simulate_path(10, 2) == 0

đường đi: 9 -> 7 -> 5 -> 3 -> 2 | chi phí: 1
đường đi: 10 -> 8 -> 6 -> 4 -> 2 | chi phí: 0


`map(str, path)` áp `str` lên từng phần tử; `" -> ".join(...)` nối các chuỗi bằng dấu `->`.

## 7. Code nộp LeetCode

### 7.1. Bản chính — đếm chẵn/lẻ

In [15]:
from typing import List

class Solution:
    def minCostToMoveChips(self, position: List[int]) -> int:
        odd = 0
        for p in position:
            if p % 2 == 1:
                odd += 1
        even = len(position) - odd
        return min(odd, even)

sol = Solution()
print(sol.minCostToMoveChips([1, 2, 3]), sol.minCostToMoveChips([2, 2, 2, 3, 3]), sol.minCostToMoveChips([1, 1000000000]))

1 2 1


### 7.2. Bản một dòng

In [16]:
class SolutionShort:
    def minCostToMoveChips(self, position: List[int]) -> int:
        odd = sum(p & 1 for p in position)
        return min(odd, len(position) - odd)

assert SolutionShort().minCostToMoveChips([2, 2, 2, 3, 3]) == 2
print("SolutionShort OK |", 7 & 1, 8 & 1)

SolutionShort OK | 1 0


`p & 1` là phép **AND bit**: lấy bit cuối của `p` — bằng 1 nếu lẻ, 0 nếu chẵn. Kết quả giống `p % 2` với số không âm.

### 7.3. Bản brute force (đúng, `O(n²)`, chỉ để tham khảo)

In [17]:
class SolutionBrute:
    def minCostToMoveChips(self, position: List[int]) -> int:
        return min(sum(abs(p - t) % 2 for p in position) for t in position)

assert SolutionBrute().minCostToMoveChips([1, 2, 3]) == 1
print("SolutionBrute OK")

SolutionBrute OK


## 8. Vì sao đúng?

**Bổ đề 1 (chi phí một chip).** Đưa chip từ `p` về `t` tốn ít nhất `|p - t| % 2`, và đạt được đúng con số đó.
- *Cận dưới:* bước 2 không đổi parity; muốn từ parity của `p` sang parity của `t` khác nhau thì phải dùng **ít nhất một** bước 1 (mỗi bước 1 đổi parity). Nếu cùng parity thì cận dưới là 0.
- *Đạt được:* đi các bước 2 miễn phí về phía `t` cho tới khi cách ≤ 1, rồi (nếu cần) một bước 1 (xem `simulate_path`).

**Bổ đề 2 (các chip độc lập).** Mỗi chip di chuyển riêng, chi phí cộng lại → tổng chi phí với đích `t` là `#{chip khác parity với t}`.

**Kết luận.** Đích chẵn → trả `odd`; đích lẻ → trả `even`. Luôn tồn tại đích của mỗi loại (ví dụ 2 và 1, hoặc chính vị trí một chip). Vậy đáp án = `min(even, odd)`. ∎

Điều này cũng giải thích vì sao brute force chỉ cần thử `t ∈ position`: nếu nhóm nào có chip thì vị trí của chip đó đại diện cho parity đó; nhóm rỗng thì đáp án là 0 và đích nằm ở nhóm còn lại.

## 9. Độ phức tạp và so sánh

Gọi `n = len(position)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Brute force mọi đích trong `position` | `O(n²)` | `O(1)` | dễ nghĩ |
| Thử 2 đích (1 và 2) | `O(n)` | `O(1)` | cầu nối |
| Đếm chẵn/lẻ | `O(n)` | `O(1)` | lời giải chuẩn |

Đặc biệt: độ phức tạp **không phụ thuộc** giá trị vị trí (tới `10^9`) — không bao giờ mô phỏng từng bước di chuyển.

## 10. Bộ test

Ngoài các ca thường, ta xây một **oracle độc lập**: với trục số nhỏ, tìm chi phí nhỏ nhất đưa mỗi chip về đích bằng **Dijkstra 0-1 BFS** trên đồ thị (cạnh ±2 trọng số 0, ±1 trọng số 1). Nếu công thức parity khớp oracle này thì ta tin chắc vào lập luận.

In [18]:
from collections import deque

def shortest_cost(src: int, dst: int, lo: int, hi: int) -> int:
    # 0-1 BFS: cạnh trọng số 0 đẩy vào đầu hàng đợi, trọng số 1 đẩy vào cuối
    INF = float("inf")
    dist = {src: 0}
    dq = deque([src])
    while dq:
        u = dq.popleft()
        for step, w in ((2, 0), (-2, 0), (1, 1), (-1, 1)):
            v = u + step
            if lo <= v <= hi and dist[u] + w < dist.get(v, INF):
                dist[v] = dist[u] + w
                if w == 0:
                    dq.appendleft(v)
                else:
                    dq.append(v)
    return dist[dst]

def min_cost_oracle(position: list[int]) -> int:
    lo, hi = min(position) - 2, max(position) + 2
    return min(sum(shortest_cost(p, t, lo, hi) for p in position) for t in range(lo, hi + 1))

print(shortest_cost(9, 2, 0, 12), min_cost_oracle([1, 2, 3]), min_cost_oracle([2, 2, 2, 3, 3]))

1 1 2


`deque` (double-ended queue) cho phép thêm/lấy ở **cả hai đầu** trong `O(1)`: `appendleft`, `append`, `popleft`.

In [19]:
impls = {
    "brute": min_cost_brute,
    "parity": min_cost_parity,
    "two_targets": min_cost_two_targets,
    "oneliner": min_cost_oneliner,
    "Solution": Solution().minCostToMoveChips,
    "SolShort": SolutionShort().minCostToMoveChips,
    "SolBrute": SolutionBrute().minCostToMoveChips,
}
cases = [
    ([1, 2, 3], 1),
    ([2, 2, 2, 3, 3], 2),
    ([1, 1000000000], 1),
    ([5], 0),                          # 1 chip
    ([4, 4, 4], 0),                    # đã ở cùng chỗ
    ([1, 3, 5, 7], 0),                 # cùng parity, khác chỗ
    ([1, 2], 1),
    ([1, 2, 4, 6, 8], 1),
    ([10**9, 10**9 - 1, 10**9 - 2, 1], 2),
]
for pos, expected in cases:
    for name, f in impls.items():
        assert f(pos) == expected, (name, pos)
    print(f"{str(pos):40} -> {expected} PASS")

[1, 2, 3]                                -> 1 PASS
[2, 2, 2, 3, 3]                          -> 2 PASS
[1, 1000000000]                          -> 1 PASS
[5]                                      -> 0 PASS
[4, 4, 4]                                -> 0 PASS
[1, 3, 5, 7]                             -> 0 PASS
[1, 2]                                   -> 1 PASS
[1, 2, 4, 6, 8]                          -> 1 PASS
[1000000000, 999999999, 999999998, 1]    -> 2 PASS


In [20]:
import random

rng = random.Random(1217)
for _ in range(200):
    pos = [rng.randint(1, 12) for _ in range(rng.randint(1, 6))]
    want = min_cost_oracle(pos)
    for name, f in impls.items():
        assert f(pos) == want, (name, pos, want)
print("200 test ngẫu nhiên: mọi cách khớp oracle 0-1 BFS")

200 test ngẫu nhiên: mọi cách khớp oracle 0-1 BFS


## 11. Lỗi người mới hay gặp

### Lỗi 1: Tính chi phí theo khoảng cách thật

In [21]:
def wrong_distance(position):
    t = sorted(position)[len(position) // 2]        # chọn trung vị như bài "dời về 1 điểm"
    return sum(abs(p - t) for p in position)        # SAI: coi mỗi đơn vị đều tốn phí

print("sai :", wrong_distance([1, 1000000000]), "| đúng:", min_cost_parity([1, 1000000000]))

sai : 999999999 | đúng: 1


Vì sao sai: bước ±2 miễn phí nên khoảng cách xa gần không quan trọng. Đây là cái bẫy do bài 462 (Minimum Moves to Equal Array Elements II) có công thức trung vị. Đọc kỹ chi phí!

### Lỗi 2: Đếm số **vị trí khác nhau** thay vì số **chip**

In [22]:
def wrong_distinct(position):
    s = set(position)                                # SAI: gộp các chip cùng chỗ
    odd = sum(p % 2 for p in s)
    return min(odd, len(s) - odd)

print("sai :", wrong_distinct([2, 2, 2, 3, 3]), "| đúng:", min_cost_parity([2, 2, 2, 3, 3]))

sai : 1 | đúng: 2


Hai chip ở vị trí 3 mỗi chip tốn 1 → 2, không phải 1.

### Lỗi 3: Trả `max(even, odd)` hoặc `odd` cố định

Luôn dồn về đích cùng parity với **nhóm đông hơn** → trả phí cho **nhóm ít hơn** → `min`.

### Lỗi 4: Mô phỏng từng bước di chuyển

Vòng `while` di chuyển chip từng bước 2 với `position` tới `10^9` → tới `5·10^8` vòng cho một chip → TLE. Chỉ cần `% 2`.

## 12. Mẹo nhận dạng pattern

- Có một thao tác **miễn phí** (hoặc rất rẻ) làm thay đổi giá trị theo **bước cố định k** → giá trị chỉ quan trọng **modulo k** (ở đây k = 2 → chẵn lẻ).
- Đề cho giá trị rất lớn (`10^9`) nhưng `n` nhỏ → thường không cần mô phỏng; tìm **bất biến**.
- "Dồn mọi thứ về một điểm" → thử nghĩ: đích tối ưu thuộc tập nhỏ nào? (trung vị, một trong các điểm, hay một lớp đồng dư…).

Bài cùng họ: 1342 Number of Steps to Reduce a Number to Zero (parity), 2032/2733 các bài parity, 453 / 462 Minimum Moves to Equal Array Elements (I, II — so sánh để thấy khác biệt), 2033 Minimum Operations to Make a Uni-Value Grid (modulo `x`).

## 13. Biến thể

### 13.1. Bước miễn phí là ±k (thay vì ±2), bước ±1 tốn 1

Lúc này vị trí chỉ quan trọng theo `p % k`. Đưa chip từ lớp dư `r` về lớp dư `c` tốn `min((r - c) % k, (c - r) % k)` bước 1. Thử mọi lớp đích `c`.

In [23]:
def min_cost_mod_k(position: list[int], k: int) -> int:
    best = float("inf")
    for c in range(k):
        total = 0
        for p in position:
            r = p % k
            total += min((r - c) % k, (c - r) % k)
        best = min(best, total)
    return best

print(min_cost_mod_k([1, 2, 3], 2), min_cost_mod_k([1, 2, 3], 3), min_cost_mod_k([0, 4, 8, 5], 4))

1 2 1


Với `k = 2` công thức trở về đúng bài gốc. Với `k = 3`, `[1, 2, 3]` cần 2 (lớp dư 0, 1, 2 — dồn về lớp giữa).

### 13.2. Bước ±1 tốn `c1`, bước ±2 tốn `c2` (tổng quát)

Nếu `c2 > 2·c1` thì đi hai bước 1 rẻ hơn một bước 2 → mọi thứ quay về khoảng cách thật; còn nếu `c2 == 0` thì là bài gốc. Đây là ví dụ cho thấy **giả thiết miễn phí** là chìa khoá — đổi giả thiết là đổi thuật toán.

### 13.3. Trả về **vị trí đích** (một vị trí có chip) đạt chi phí nhỏ nhất

In [24]:
def best_target(position: list[int]) -> int:
    odd = sum(p % 2 for p in position)
    even = len(position) - odd
    want_parity = 0 if even >= odd else 1
    return next(p for p in position if p % 2 == want_parity)

print(best_target([2, 2, 2, 3, 3]), best_target([1, 2, 3]))

2 1


`next(generator)` lấy phần tử **đầu tiên** mà generator sinh ra — ở đây là chip đầu tiên có parity mong muốn.

## 14. Bài tập (TODO — không gọi khi chạy)

**Bài 1 (tính tay).** `position = [3, 3, 6, 6, 6, 9, 12]`. Bao nhiêu chip chẵn, bao nhiêu lẻ? Đáp án?

**Bài 2 (dự đoán output).** `abs(7 - 2) % 2`, `sum(p % 2 for p in [1, 2, 3, 4, 5])`, `(10**9) & 1`.

**Bài 3 (điền TODO).**

In [25]:
def exercise_min_cost(position: list[int]) -> int:
    even = odd = 0
    # TODO 1: duyệt p, tăng even nếu p chẵn, ngược lại tăng odd
    # TODO 2: trả min(even, odd)
    pass

# assert exercise_min_cost([2, 2, 2, 3, 3]) == 2
# assert exercise_min_cost([1, 1000000000]) == 1

**Bài 4 (viết lại).** Viết `exercise_cost_to_target(position, t)` trả chi phí dồn **về đúng vị trí `t`**.

In [26]:
def exercise_cost_to_target(position: list[int], t: int) -> int:
    # TODO
    pass

# assert exercise_cost_to_target([1, 2, 3], 1) == 1
# assert exercise_cost_to_target([1, 2, 3], 2) == 2

**Bài 5 (biến thể).** Bước miễn phí là ±3, bước ±1 tốn 1. Viết hàm tính chi phí nhỏ nhất (gợi ý: lớp dư modulo 3).

In [27]:
def exercise_min_cost_mod3(position: list[int]) -> int:
    # TODO
    pass

# assert exercise_min_cost_mod3([1, 2, 3]) == 2
# assert exercise_min_cost_mod3([3, 6, 9, 1]) == 1

**Bài 6 (mini challenge).** Ngoài chi phí nhỏ nhất, trả về **số vị trí đích khác nhau trong đoạn `[lo, hi]`** đạt chi phí nhỏ nhất (đếm bằng công thức, không duyệt từng số).

In [28]:
def exercise_count_best_targets(position: list[int], lo: int, hi: int) -> int:
    # TODO
    pass

# assert exercise_count_best_targets([2, 2, 2, 3, 3], 1, 10) == 5      # các đích chẵn 2,4,6,8,10
# assert exercise_count_best_targets([1, 2], 1, 10) == 10              # chẵn hay lẻ đều tốn 1

## 15. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Phân nhóm theo `% 2` | đếm từng nhóm | nhóm ít hơn trả phí |
| 2 | khoảng cách rồi `% 2` | `True`/1 cộng được | `& 1` là bit cuối |
| 3 | if/else theo `p % 2` | `even += 1` / `odd += 1` | `return min(even, odd)` |
| 4 | Mỗi chip độc lập | chi phí chip = `abs(p - t) % 2` | `sum(...)` |
| 5 | Chỉ quan trọng `p % 3` | thử 3 lớp đích | chi phí lớp: `min((r - c) % 3, (c - r) % 3)` |
| 6 | Đích tốt là cả một lớp parity | đếm số chẵn / số lẻ trong `[lo, hi]` | số chẵn trong `[lo, hi]` = `hi // 2 - (lo - 1) // 2`; nếu `even == odd` thì cả hai lớp |

## 16. Đáp án tham khảo

**Bài 1:** chẵn: 6, 6, 6, 12 → 4; lẻ: 3, 3, 9 → 3 → đáp án **3**.

**Bài 2:** `1`, `3`, `0`.

In [29]:
def answer_min_cost(position):
    even = odd = 0
    for p in position:
        if p % 2 == 0:
            even += 1
        else:
            odd += 1
    return min(even, odd)

def answer_cost_to_target(position, t):
    return sum(abs(p - t) % 2 for p in position)

def answer_min_cost_mod3(position):
    return min_cost_mod_k(position, 3)

def answer_count_best_targets(position, lo, hi):
    odd = sum(p % 2 for p in position)
    even = len(position) - odd
    evens_in_range = hi // 2 - (lo - 1) // 2
    odds_in_range = (hi - lo + 1) - evens_in_range
    if even > odd:
        return evens_in_range          # dồn về đích chẵn tốt hơn
    if odd > even:
        return odds_in_range
    return hi - lo + 1                 # bằng nhau: mọi đích đều tốt

assert answer_min_cost([3, 3, 6, 6, 6, 9, 12]) == 3
assert (abs(7 - 2) % 2, sum(p % 2 for p in [1, 2, 3, 4, 5]), (10**9) & 1) == (1, 3, 0)
assert answer_min_cost([2, 2, 2, 3, 3]) == 2 and answer_min_cost([1, 1000000000]) == 1
assert answer_cost_to_target([1, 2, 3], 1) == 1 and answer_cost_to_target([1, 2, 3], 2) == 2
assert answer_min_cost_mod3([1, 2, 3]) == 2 and answer_min_cost_mod3([3, 6, 9, 1]) == 1
assert answer_count_best_targets([2, 2, 2, 3, 3], 1, 10) == 5
assert answer_count_best_targets([1, 2], 1, 10) == 10
# kiểm tra bài 6 bằng cách duyệt thật
for pos, lo, hi in [([2, 2, 2, 3, 3], 1, 10), ([1, 3, 4], 2, 9), ([5], 3, 3)]:
    costs = [answer_cost_to_target(pos, t) for t in range(lo, hi + 1)]
    assert answer_count_best_targets(pos, lo, hi) == costs.count(min(costs))
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


## 17. Template nhớ nhanh và checklist submit

```python
odd = sum(p % 2 for p in position)
return min(odd, len(position) - odd)
```

**Câu thần chú:** *"Bước 2 miễn phí → chỉ còn chẵn/lẻ. Nhóm ít hơn trả phí."*

### Checklist trước khi submit
- [ ] Không dùng khoảng cách thật `abs(p - t)` làm chi phí?
- [ ] Đếm theo **từng chip**, không gộp vị trí trùng bằng `set`?
- [ ] Trả `min(even, odd)` chứ không phải `max`?
- [ ] Không mô phỏng từng bước (vị trí tới `10^9`)?
- [ ] Tên method `minCostToMoveChips(self, position)`?